# BuzzSpot Challenge — Pipeline End-to-End

**CVPPA @ ECCV 2026 — Détection de pollinisateurs en vidéo**

Ce notebook couvre l'intégralité du pipeline :
1. Téléchargement & extraction du dataset BuzzSet
2. Exploration & statistiques
3. (Optionnel) Extraction des masques SAM 2 pour copy-paste
4. Entraînement (RF-DETR + fusion temporelle)
5. Inférence tuilée + TTA + WBF
6. Évaluation COCO (mAP@0.5:0.95)
7. Génération de la soumission `.zip`

---
**Repo** : https://github.com/hashirama21/buzzspot-gik  
**Dataset** : https://phenoroam.phenorob.de/file-uploader/download/public/35261367058-BuzzSet_challenge.zip

## 0 — Environnement

In [ ]:
import sys, os

# Détection Colab vs local
IN_COLAB = 'google.colab' in sys.modules
print('Colab :', IN_COLAB)

## 1 — Installation des dépendances

In [ ]:
# Ajuster l'index si nécessaire : cu118 pour CUDA 11.8, cpu pour CPU only
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -q \
    rfdetr \
    albumentations>=1.4.0 \
    opencv-python-headless>=4.9.0 \
    pycocotools>=2.0.7 \
    hydra-core>=1.3.2 \
    omegaconf>=2.3.0 \
    scipy>=1.13.0 \
    numpy>=1.26.0 \
    pandas>=2.2.0 \
    wandb \
    tqdm \
    rich \
    matplotlib \
    seaborn

# !pip install -q groundingdino-py segment-anything-2

print('✓ Installation terminée')

## 2 — Clonage du repo

In [ ]:
REPO_URL  = 'https://github.com/hashirama21/buzzspot-gik.git'
REPO_DIR  = 'buzzspot-gik'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print('Répertoire courant :', os.getcwd())

## 3 — Téléchargement et extraction du dataset BuzzSet

In [ ]:
import zipfile
import urllib.request
from pathlib import Path
from tqdm.notebook import tqdm

DATASET_URL  = 'https://phenoroam.phenorob.de/file-uploader/download/public/35261367058-BuzzSet_challenge.zip'
ZIP_PATH     = Path('data/BuzzSet_challenge.zip')
EXTRACT_ROOT = Path('data/')

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    print(f'Téléchargement depuis {DATASET_URL} …')

    class _ProgressBar(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)

    with _ProgressBar(unit='B', unit_scale=True, unit_divisor=1024, miniters=1,
                      desc='BuzzSet') as pb:
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH, reporthook=pb.update_to)
    print(f'✓ Archive sauvegardée : {ZIP_PATH}  ({ZIP_PATH.stat().st_size / 1e6:.0f} MB)')
else:
    print(f'✓ Archive déjà présente : {ZIP_PATH}')

# Extraction
BUZZSET_DIR = EXTRACT_ROOT / 'buzzset'
if not BUZZSET_DIR.exists():
    print('Extraction …')
    with zipfile.ZipFile(ZIP_PATH) as z:
        members = z.namelist()
        for m in tqdm(members, desc='Extraction'):
            z.extract(m, EXTRACT_ROOT)
    print('✓ Extraction terminée')
else:
    print('✓ Dataset déjà extrait')

In [ ]:
# Vérification de la structure attendue
for split in ('train', 'valid', 'test'):
    ann = EXTRACT_ROOT / 'buzzset' / 'annotations' / f'{split}.json'
    img = EXTRACT_ROOT / 'buzzset' / split
    status = '✓' if ann.exists() and img.exists() else '✗'
    print(f'{status}  {split:5s} — annotations: {ann.exists()}, images: {img.exists()}')

## 4 — Exploration du dataset

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

SPLITS = {}
for split in ('train', 'valid', 'test'):
    ann_path = EXTRACT_ROOT / 'buzzset' / 'annotations' / f'{split}.json'
    if ann_path.exists():
        with open(ann_path) as f:
            SPLITS[split] = json.load(f)

# Statistiques générales
for split, data in SPLITS.items():
    n_imgs = len(data['images'])
    n_anns = len(data.get('annotations', []))
    print(f'{split:6s} : {n_imgs:5d} images, {n_anns:6d} annotations')

In [ ]:
# Distribution des classes dans le train set
CLASS_NAMES = {1: 'bee', 2: 'bumblebee', 3: 'hoverfly', 4: 'moth'}

if 'train' in SPLITS:
    train_data = SPLITS['train']
    counts = {name: 0 for name in CLASS_NAMES.values()}
    blur_count = occ_count = 0

    for ann in train_data['annotations']:
        cls = CLASS_NAMES.get(ann['category_id'], 'unknown')
        counts[cls] = counts.get(cls, 0) + 1
        attrs = ann.get('attributes', {})
        blur_count += attrs.get('blur', 0)
        occ_count  += attrs.get('occlusion', 0)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Barplot classes
    ax = axes[0]
    bars = ax.bar(list(counts.keys()), list(counts.values()),
                  color=sns.color_palette('muted', len(counts)))
    ax.bar_label(bars)
    ax.set_title('Distribution des classes (train)')
    ax.set_ylabel('Nombre d\'instances')

    # Attributs
    ax = axes[1]
    total = len(train_data['annotations'])
    ax.bar(['Blur', 'Occlusion', 'Clean'],
           [blur_count, occ_count, total - blur_count - occ_count],
           color=['#e74c3c', '#e67e22', '#2ecc71'])
    ax.set_title('Attributs de dégradation')
    ax.set_ylabel('Instances')

    plt.tight_layout()
    plt.savefig('data_exploration.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Total train annotations : {total}')
    print(f'Blur : {blur_count} ({100*blur_count/total:.1f}%)')
    print(f'Occlusion : {occ_count} ({100*occ_count/total:.1f}%)')

In [ ]:
# Distribution des tailles de boîtes (pixels²)
if 'train' in SPLITS:
    areas = [ann['area'] for ann in SPLITS['train']['annotations']]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(np.sqrt(areas), bins=60, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.axvline(32,  color='r', linestyle='--', label='Seuil small (32px)')
    ax.axvline(96,  color='orange', linestyle='--', label='Seuil medium (96px)')
    ax.set_xlabel('√aire (pixels)')
    ax.set_ylabel('Fréquence')
    ax.set_title('Distribution des tailles d\'instances')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualisation de quelques images annotées
import cv2
import random

COLORS = {1: (52,152,219), 2: (231,76,60), 3: (46,204,113), 4: (155,89,182)}

def show_annotated_sample(split='train', n=4, seed=42):
    data    = SPLITS[split]
    id2img  = {img['id']: img for img in data['images']}
    id2anns = {}
    for ann in data.get('annotations', []):
        id2anns.setdefault(ann['image_id'], []).append(ann)

    rng      = random.Random(seed)
    img_ids  = [iid for iid in id2anns if id2anns[iid]]
    sample   = rng.sample(img_ids, min(n, len(img_ids)))

    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 5))
    if len(sample) == 1:
        axes = [axes]

    for ax, img_id in zip(axes, sample):
        meta     = id2img[img_id]
        img_path = EXTRACT_ROOT / 'buzzset' / split / meta['file_name']
        img_bgr  = cv2.imread(str(img_path))
        if img_bgr is None:
            ax.set_title('Image introuvable')
            continue
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()

        for ann in id2anns.get(img_id, []):
            x, y, w, h = [int(v) for v in ann['bbox']]
            cls    = ann['category_id']
            color  = COLORS.get(cls, (255, 255, 0))
            label  = CLASS_NAMES.get(cls, '?')
            cv2.rectangle(img_rgb, (x, y), (x+w, y+h), color, 2)
            cv2.putText(img_rgb, label, (x, max(y-4, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        ax.imshow(img_rgb)
        ax.set_title(f'{meta["file_name"].split("/")[-1]}')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_annotated_sample('train', n=4)

## 5 — Preprocessing : extraction des masques SAM 2 (optionnel)

> Cette étape génère les crops RGBA utilisés par la copie-collage pour booster
> les classes rares (moth, hoverfly). Elle nécessite `sam2` installé et un checkpoint.
> **Peut être sautée** — le training fonctionnera sans copy-paste.

In [ ]:
SAM2_CKPT    = 'weights/sam2_hiera_large.pt'   # Adapter si nécessaire
MASKS_OUT    = 'data/sam2_masks/'
RUN_SAM2_STEP = False  # Passer à True si sam2 est installé et le checkpoint disponible

if RUN_SAM2_STEP:
    !python scripts/extract_sam2_masks.py \
        --ann      data/buzzset/annotations/train.json \
        --img-dir  data/buzzset/train/ \
        --out-dir  {MASKS_OUT} \
        --sam2-ckpt {SAM2_CKPT} \
        --classes moth hoverfly
else:
    print('⏭  Étape SAM2 ignorée (RUN_SAM2_STEP=False)')
    # Désactiver la copy-paste dans la config si pas de masques
    import subprocess
    # Sera géré par la config via augmentations.train.copy_paste.enabled=false

## 6 — Entraînement

In [ ]:
# Configuration de l'expérience
EXPERIMENT_NAME = 'rfdetr_temporal_v1'
EPOCHS          = 50          # Réduire pour un test rapide (ex: 3)
BATCH_SIZE      = 8           # Réduire si OOM (ex: 4)
NUM_WORKERS     = 4
COPY_PASTE_ON   = False       # Mettre True si les masques SAM2 ont été générés

# Overrides Hydra
overrides = [
    f'training.experiment_name={EXPERIMENT_NAME}',
    f'training.epochs={EPOCHS}',
    f'training.batch_size={BATCH_SIZE}',
    f'data.num_workers={NUM_WORKERS}',
    f'data.root=data/buzzset/',
    f'data.train_ann=data/buzzset/annotations/train.json',
    f'data.val_ann=data/buzzset/annotations/valid.json',
    f'augmentations.train.copy_paste.enabled={str(COPY_PASTE_ON).lower()}',
]

print('Lancement de l\'entraînement avec les paramètres :')
for ov in overrides:
    print(' ', ov)

In [ ]:
# Lancement de l'entraînement
override_str = ' '.join(overrides)
!python src/train.py {override_str}

In [ ]:
# Visualisation des courbes d'entraînement (si wandb non utilisé)
# Si wandb est configuré, consulter directement https://wandb.ai
import glob

ckpt_path = f'outputs/{EXPERIMENT_NAME}/best.pth'
if os.path.exists(ckpt_path):
    import torch
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'✓ Meilleur checkpoint — epoch {ckpt["epoch"]},')
    print(f'  best_metric = {ckpt["best_metric"]:.4f}')
else:
    print('✗ Checkpoint best.pth introuvable')

## 7 — Inférence sur le jeu de test

In [ ]:
WEIGHTS_PATH  = f'outputs/{EXPERIMENT_NAME}/best.pth'
PREDICTIONS   = f'submissions/predictions_{EXPERIMENT_NAME}.json'
SCORE_THR     = 0.3

!python scripts/infer.py \
    --weights   {WEIGHTS_PATH} \
    --test-ann  data/buzzset/annotations/test.json \
    --test-dir  data/buzzset/test/ \
    --out       {PREDICTIONS} \
    --score-thr {SCORE_THR} \
    --zip

In [ ]:
# Résumé des prédictions
from pathlib import Path
import json

pred_path = Path(PREDICTIONS)
if pred_path.exists():
    with open(pred_path) as f:
        preds = json.load(f)

    print(f'Total prédictions : {len(preds)}')
    cls_counts = {}
    for p in preds:
        cls_counts[p['category_id']] = cls_counts.get(p['category_id'], 0) + 1
    for cat_id, cnt in sorted(cls_counts.items()):
        print(f'  {CLASS_NAMES.get(cat_id, cat_id):12s} : {cnt:6d}')

    scores = [p['score'] for p in preds]
    print(f'\nScore — min: {min(scores):.3f}, mean: {np.mean(scores):.3f}, max: {max(scores):.3f}')
else:
    print('✗ Fichier de prédictions introuvable')

## 8 — Évaluation COCO sur la validation

In [ ]:
import torch
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

from src.data import BuzzSetDataset
from src.data.augmentations import get_val_transforms
from src.models.rfdetr_temporal import RFDETRTemporal
from src.evaluation.metrics.coco_eval import BuzzSpotEvaluator
from src.train import collate_fn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE)

# Chargement du checkpoint
ckpt = torch.load(WEIGHTS_PATH, map_location=DEVICE, weights_only=False)
cfg  = OmegaConf.create(ckpt['config'])

model = RFDETRTemporal(
    num_classes=cfg.data.num_classes,
    d_model=cfg.model.hidden_dim,
    num_queries=cfg.model.num_queries,
    num_frames=cfg.data.temporal.num_frames,
    temporal_type=cfg.model.temporal_head.type,
    use_attribute_heads=cfg.model.attribute_heads.enabled,
)
model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE).eval()
print(f'✓ Modèle chargé — epoch {ckpt["epoch"]}')

In [ ]:
# Validation COCO
val_ds = BuzzSetDataset(
    ann_file='data/buzzset/annotations/valid.json',
    img_dir='data/buzzset/',
    transforms=get_val_transforms(tile_size=cfg.data.tile.size),
    num_frames=cfg.data.temporal.num_frames,
    temporal_strategy=cfg.data.temporal.strategy,
    split='val',
)

val_loader = DataLoader(
    val_ds, batch_size=16, shuffle=False,
    num_workers=2, collate_fn=collate_fn,
)

evaluator = BuzzSpotEvaluator(
    ann_file='data/buzzset/annotations/valid.json',
    class_names=list(cfg.data.class_names),
)

from torch.amp import autocast
from tqdm.notebook import tqdm

use_amp = DEVICE.type == 'cuda'

with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluation'):
        images  = batch['image'].to(DEVICE)
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in batch['target']]
        with autocast(device_type=DEVICE.type, enabled=use_amp):
            outputs = model(images)
        evaluator.update(outputs, targets)

metrics = evaluator.summarize()

print('\n' + '='*50)
print('RÉSULTATS COCO')
print('='*50)
for k, v in metrics.items():
    print(f'  {k:<25s}: {v:.4f}')

In [ ]:
# Visualisation des métriques par classe
class_keys = {cls: (metrics.get(f'AP_{cls}_50_95', 0), metrics.get(f'AP_{cls}_50', 0))
              for cls in ['bee', 'bumblebee', 'hoverfly', 'moth']}

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(class_keys))
w = 0.35
cls_names = list(class_keys.keys())
ap5095 = [class_keys[c][0] for c in cls_names]
ap50   = [class_keys[c][1] for c in cls_names]

ax.bar(x - w/2, ap5095, w, label='AP@0.5:0.95', color='steelblue')
ax.bar(x + w/2, ap50,   w, label='AP@0.5',      color='lightblue')
ax.set_xticks(x)
ax.set_xticklabels(cls_names)
ax.set_ylabel('Average Precision')
ax.set_title(f'Per-class AP — mAP@0.5:0.95 = {metrics["mAP_50_95"]:.4f}')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 9 — Génération de la soumission Codabench

In [ ]:
import zipfile
from pathlib import Path

pred_json = Path(PREDICTIONS)
zip_path  = pred_json.parent / (pred_json.stem + '.zip')

if pred_json.exists():
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_json, 'predictions.json')
    print(f'✓ Soumission prête : {zip_path}')
    print(f'  Taille : {zip_path.stat().st_size / 1024:.1f} KB')
    print('\n→ Uploader ce fichier sur https://codalab.lisn.upsaclay.fr/ ou Codabench')
else:
    print('✗ Fichier de prédictions manquant')

## 10 — (Optionnel) Pseudo-labeling + self-training

In [ ]:
RUN_PSEUDO_LABELING = False  # Nécessite groundingdino + sam2

if RUN_PSEUDO_LABELING:
    GDINO_CFG  = 'weights/GroundingDINO_SwinT_OGC.py'
    GDINO_CKPT = 'weights/groundingdino_swint_ogc.pth'
    SAM2_CKPT  = 'weights/sam2_hiera_large.pt'

    !python scripts/pseudo_label.py \
        --weights    {WEIGHTS_PATH} \
        --test-ann   data/buzzset/annotations/test.json \
        --test-dir   data/buzzset/test/ \
        --gdino-cfg  {GDINO_CFG} \
        --gdino-ckpt {GDINO_CKPT} \
        --sam2-ckpt  {SAM2_CKPT} \
        --out-ann    data/buzzset/annotations/pseudo_train.json \
        --conf       0.5 \
        --epochs     2
else:
    print('⏭  Pseudo-labeling ignoré (RUN_PSEUDO_LABELING=False)')